# 01. Контекстный пакет

В этом мини-примере агент не выглядит как «живое существо». Это обычный харнес вокруг LLM: он собирает нужный контекст, передает его в модель и потом разбирает ответ.

До кэпстоуна используем нейтральный docs-like сценарий: служба поддержки интернет-магазина, заказ, политика возврата и черновик ответа клиенту. Это близко к типовым примерам про customer support, refunds и tool use из документации LLM-платформ.

Наша задача: руками собрать маленький пакет контекста для будущего support-агента.

In [ ]:
from pprint import pprint


context_packet = {
    "task": "Подготовить ответ клиенту по возврату заказа ORD-1842",
    "instructions": [
        "Пиши кратко и проверяемо.",
        "Не добавляй факты без источника.",
        "Если политика и сообщение клиента конфликтуют, пометь это явно.",
    ],
    "sections": [
        {
            "name": "run_state",
            "trust": "internal",
            "content": "run_id=demo-001; step=draft_reply; case_id=ORD-1842",
        },
        {
            "name": "refund_policy",
            "trust": "workspace",
            "content": "Возврат доступен в течение 30 дней после доставки, если товар не использовался.",
        },
        {
            "name": "customer_message",
            "trust": "untrusted_source",
            "content": "Клиент пишет, что заказ доставлен 12 дней назад, упаковка открыта, товар не использовался.",
        },
    ],
}

pprint(context_packet, sort_dicts=False)

Главная мысль: **память агента не возникает сама**. Харнес явно решает, что положить в следующий вызов модели: инструкции, состояние запуска, политику, сообщение клиента и ограничения.

In [ ]:
def render_context(packet: dict) -> str:
    lines = [
        "# Задача",
        packet["task"],
        "",
        "# Правила",
    ]
    lines.extend(f"- {item}" for item in packet["instructions"])
    lines.append("")

    for section in packet["sections"]:
        lines.append(f"# {section['name']} [{section['trust']}]")
        lines.append(section["content"])
        lines.append("")

    return "\n".join(lines).strip()


print(render_context(context_packet))